# XGBoost Inference-Safe Tuning

## tl;dr

This notebook tests whether the local full-feature XGBoost gain can be recovered with Kaggle-legal features. It focuses on fold-safe predicted formation proxies and row-level typewell projection features.

## Context & Methods

The current best public submission is inference-safe XGBoost with public LB **14.472**. Full-feature local XGBoost scored **12.7504** on `strict_hybrid_fold`, but its top features included formation distances absent from hidden test files. This notebook replaces direct formation columns with proxy features that can be computed for hidden wells.

### Key Assumptions

- Direct formation columns are not Kaggle-safe.
- Proxy features must be built fold-safely during validation.
- Local decisions use strict folds and analog-distance stress folds before Kaggle packaging.

## Setup

In [1]:

"Imports, paths, and deterministic configuration."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parent.parent.parent

RUN_ID = "2026-06-28__xgboost_inference_safe_tuning"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
for path_item in [RUN_DIR, TABLE_DIR, REPORT_DIR]:
    path_item.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "fold_column": "strict_hybrid_fold",
    "stress_fold_column": "analog_distance_fold",
    "max_target_rows_per_well": 120,
    "random_state": 20260628,
    "current_public_baseline": 14.472,
    "xgb_full_feature_rmse": 12.7504,
    "xgb_full_feature_stress_rmse": 12.9116,
    "xgb_public_inference_safe": 14.472,
    "lightgbm_public_inference_safe": 14.687,
    "promotion_gate_vs_public": 0.05,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

print(ROOT)
print(RUN_DIR)


C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-28__xgboost_inference_safe_tuning


## Feature Construction

In [2]:

"Build sampled target-zone rows with direct formation labels retained only for proxy training diagnostics."

folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptors = pd.read_csv(TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv")
well_meta = folds.merge(descriptors, on="wellname", how="left")
well_meta = well_meta.drop_duplicates(subset=["wellname"]).reset_index(drop=True)

formation_cols = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
descriptor_cols = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]


def safe_divide(num, den):
    return num / (den + 1e-9)


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def interp_typewell_gr(typewell, tvt_values):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float), clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_target_rows(csv_path, meta_row):
    df = pd.read_csv(csv_path)
    df = df.sort_values("MD").reset_index(drop=True)
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0 or (~known_mask).sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    target = df.loc[last_known_idx + 1:].copy()
    if target.empty:
        return pd.DataFrame()
    if len(target) > CONFIG["max_target_rows_per_well"]:
        sample_idx = np.linspace(0, len(target) - 1, CONFIG["max_target_rows_per_well"]).round().astype(int)
        target = target.iloc[np.unique(sample_idx)].copy()
    typewell = load_typewell(csv_path.parent, meta_row["wellname"])
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)
    target = target.reset_index(drop=False).rename(columns={"index": "row_idx_original"})
    out = pd.DataFrame({
        "wellname": meta_row["wellname"],
        "row_idx_original": target["row_idx_original"].astype(float),
        "MD": target["MD"].astype(float),
        "X": target["X"].astype(float),
        "Y": target["Y"].astype(float),
        "Z": target["Z"].astype(float),
        "GR": target_gr.to_numpy(),
        "TVT": target["TVT"].astype(float),
        "last_known_tvt_feature": float(last["TVT"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
    })
    out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(meta_row.get("typewell_best_lag_tvt", 0.0) if pd.notna(meta_row.get("typewell_best_lag_tvt", np.nan)) else 0.0)
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy())
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy() + best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]
    for col in formation_cols:
        out[f"formation_top_{col}"] = target[col].astype(float).to_numpy()
        out[f"direct_dist_to_{col}"] = out[f"formation_top_{col}"] - out["Z"]
        out[f"direct_abs_dist_to_{col}"] = np.abs(out[f"direct_dist_to_{col}"])
    out["direct_min_abs_formation_dist"] = out[[f"direct_abs_dist_to_{col}" for col in formation_cols]].min(axis=1)
    out["direct_mean_abs_formation_dist"] = out[[f"direct_abs_dist_to_{col}" for col in formation_cols]].mean(axis=1)
    for col in [CONFIG["fold_column"], CONFIG["stress_fold_column"], "descriptor_cluster_fold", "extrapolation_fold"]:
        out[col] = int(meta_row[col])
    for col in descriptor_cols:
        out[col] = float(meta_row[col]) if pd.notna(meta_row[col]) else np.nan
    return out

rows = []
train_dir = ROOT / "data" / "raw" / "train"
for meta_row in well_meta.to_dict(orient="records"):
    csv_path = train_dir / f"{meta_row['wellname']}__horizontal_well.csv"
    if csv_path.exists():
        frame = build_target_rows(csv_path, meta_row)
        if not frame.empty:
            rows.append(frame)
data = pd.concat(rows, ignore_index=True)

safe_row_cols = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "dx_from_last", "dy_from_last", "dz_from_last",
    "horizontal_from_last", "distance_from_last", "distance_log1p",
]
safe_feature_cols = [col for col in [*safe_row_cols, *descriptor_cols] if col in data.columns]
typewell_feature_cols = [
    "tvt_linear_proxy", "typewell_gr_linear_tvt", "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag",
]
typewell_feature_cols = [col for col in typewell_feature_cols if col in data.columns]
direct_formation_feature_cols = [
    *[f"direct_dist_to_{col}" for col in formation_cols],
    *[f"direct_abs_dist_to_{col}" for col in formation_cols],
    "direct_min_abs_formation_dist", "direct_mean_abs_formation_dist",
]

feature_preview_path = TABLE_DIR / "model__xgboost_inference_safe_tuning_feature_preview__2026-06-28.csv"
data.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print(data.shape)
print("safe_features", len(safe_feature_cols))
print("typewell_features", typewell_feature_cols)
data.head()


(92760, 71)
safe_features 35
typewell_features ['tvt_linear_proxy', 'typewell_gr_linear_tvt', 'typewell_gr_lag_best', 'gr_minus_typewell_linear', 'gr_minus_typewell_bestlag']


,wellname,row_idx_original,MD,X,Y,Z,GR,TVT,last_known_tvt_feature,last_known_md,last_known_x,last_known_y,last_known_z,first_md,target_delta,md_from_last_known,md_from_start,eval_progress,dx_from_last,dy_from_last,dz_from_last,horizontal_from_last,distance_from_last,distance_log1p,tvt_linear_proxy,typewell_gr_linear_tvt,typewell_gr_lag_best,gr_minus_typewell_linear,gr_minus_typewell_bestlag,formation_top_ANCC,direct_dist_to_ANCC,direct_abs_dist_to_ANCC,formation_top_ASTNU,direct_dist_to_ASTNU,direct_abs_dist_to_ASTNU,formation_top_ASTNL,direct_dist_to_ASTNL,direct_abs_dist_to_ASTNL,formation_top_EGFDU,direct_dist_to_EGFDU,direct_abs_dist_to_EGFDU,formation_top_EGFDL,direct_dist_to_EGFDL,direct_abs_dist_to_EGFDL,formation_top_BUDA,direct_dist_to_BUDA,direct_abs_dist_to_BUDA,direct_min_abs_formation_dist,direct_mean_abs_formation_dist,strict_hybrid_fold,analog_distance_fold,descriptor_cluster_fold,extrapolation_fold,known_rows,known_tvt_span,known_md_span,known_gr_mean_last300,known_gr_std_last300,known_gr_slope_last300,xy_span_known,z_drop_known,typewell_corr_last300,typewell_mae_last300,typewell_best_lag_corr,typewell_best_lag_tvt,typewell_best_lag_mae,selected_analog_distance,extrapolation_stress_score,analog_stress_score,combined_stress_score,descriptor_cluster
0,000d7d20,1442.0,12909.0,2983537.06,1070212.72,-9735.07,106.984057,11747.38,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.01,1.0,1442.0,0.008333,-0.03,1.00,0.01,1.000450,1.000500,0.693397,11748.37,99.0484,75.8824,7.935657,31.101657,-9360.95,374.12,374.12,-9535.00,200.07,200.07,-9562.78,172.29,172.29,-9636.13,98.94,98.94,-9671.10,63.97,63.97,-9811.49,-76.42,76.42,63.97,164.301667,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
1,000d7d20,1474.0,12941.0,2983536.12,1070244.70,-9734.59,75.985302,11747.51,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.14,33.0,1474.0,0.016667,-0.97,32.98,0.49,32.994262,32.997900,3.526299,11780.37,100.2336,94.0140,-24.248298,-18.028698,-9360.34,374.25,374.25,-9534.39,200.20,200.20,-9562.17,172.42,172.42,-9635.52,99.07,99.07,-9670.49,64.10,64.10,-9810.88,-76.29,76.29,64.10,164.388333,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
2,000d7d20,1506.0,12973.0,2983535.21,1070276.68,-9733.80,74.180426,11747.61,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.24,65.0,1506.0,0.025000,-1.88,64.96,1.28,64.987199,64.999803,4.189652,11812.37,91.7860,83.9332,-17.605574,-9.752774,-9359.45,374.35,374.35,-9533.50,200.30,200.30,-9561.28,172.52,172.52,-9634.63,99.17,99.17,-9669.60,64.20,64.20,-9809.99,-76.19,76.19,64.20,164.455000,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
3,000d7d20,1539.0,13006.0,2983534.16,1070309.64,-9732.81,82.067736,11748.38,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,1.01,98.0,1539.0,0.033333,-2.93,97.92,2.27,97.963826,97.990123,4.595020,11845.37,29.7608,143.7476,52.306936,-61.679864,-9357.69,375.12,375.12,-9531.74,201.07,201.07,-9559.52,173.29,173.29,-9632.87,99.94,99.94,-9667.84,64.97,64.97,-9808.23,-75.42,75.42,64.97,164.968333,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
4,000d7d20,1571.0,13038.0,2983532.98,1070341.61,-9731.76,113.887709,11749.04,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,1.67,130.0,1571.0,0.041667,-4.11,129.89,3.32,129.955008,129.997410,4.875178,11877.37,NaN,29.0828,NaN,84.804909,-9355.98,375.78,375.78,-9530.03,201.73,201.73,-9557.81,173.95,173.95,-9631.16,100.60,100.60,-9666.13,65.63,65.63,-9806.52,-74.76,74.76,65.63,165.408333,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0

## Fold-Safe Formation Proxy Features

In [3]:
"Create predicted formation-distance proxies with models trained inside each validation fold."

proxy_base_cols = [col for col in [*safe_feature_cols, *typewell_feature_cols] if col in data.columns]
proxy_quality_rows = []
proxy_params = {
    "objective": "reg:squarederror",
    "n_estimators": 220,
    "learning_rate": 0.05,
    "max_depth": 4,
    "min_child_weight": 80,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 8.0,
    "reg_alpha": 0.10,
    "tree_method": "hist",
    "random_state": CONFIG["random_state"],
    "n_jobs": 4,
    "verbosity": 0,
}

for fold_col in [CONFIG["fold_column"], CONFIG["stress_fold_column"]]:
    for col in formation_cols:
        data[f"proxy_top_{col}__{fold_col}"] = np.nan
    for fold in sorted(data[fold_col].unique()):
        valid_mask = data[fold_col].eq(fold)
        train_mask = ~valid_mask
        X_proxy_train_all = data.loc[train_mask, proxy_base_cols].replace([np.inf, -np.inf], np.nan)
        X_proxy_valid = data.loc[valid_mask, proxy_base_cols].replace([np.inf, -np.inf], np.nan)
        for col in formation_cols:
            target_col = f"formation_top_{col}"
            proxy_col = f"proxy_top_{col}__{fold_col}"
            label_train_mask = train_mask & data[target_col].notna()
            label_valid_mask = valid_mask & data[target_col].notna()
            global_median = float(data[target_col].median())
            if label_train_mask.sum() >= 20:
                proxy_model = XGBRegressor(**proxy_params)
                proxy_model.fit(
                    data.loc[label_train_mask, proxy_base_cols].replace([np.inf, -np.inf], np.nan),
                    data.loc[label_train_mask, target_col].astype(float),
                    verbose=False,
                )
                pred_train = proxy_model.predict(X_proxy_train_all)
                pred_valid = proxy_model.predict(X_proxy_valid)
                train_index = data.loc[train_mask].index
                data.loc[train_mask, proxy_col] = data.loc[train_mask, proxy_col].fillna(pd.Series(pred_train, index=train_index))
                data.loc[valid_mask, proxy_col] = pred_valid
                if label_valid_mask.sum() > 0:
                    actual_valid = data.loc[label_valid_mask, target_col].astype(float).to_numpy()
                    scored_pred = pd.Series(pred_valid, index=data.loc[valid_mask].index).loc[label_valid_mask].to_numpy()
                    proxy_quality_rows.append({
                        "fold_column": fold_col,
                        "fold": int(fold),
                        "formation": col,
                        "n_train_labels": int(label_train_mask.sum()),
                        "n_valid_labels": int(label_valid_mask.sum()),
                        "top_rmse": float(mean_squared_error(actual_valid, scored_pred) ** 0.5),
                        "top_mae": float(mean_absolute_error(actual_valid, scored_pred)),
                    })
            else:
                data.loc[train_mask, proxy_col] = data.loc[train_mask, proxy_col].fillna(global_median)
                data.loc[valid_mask, proxy_col] = global_median
                proxy_quality_rows.append({
                    "fold_column": fold_col,
                    "fold": int(fold),
                    "formation": col,
                    "n_train_labels": int(label_train_mask.sum()),
                    "n_valid_labels": int(label_valid_mask.sum()),
                    "top_rmse": np.nan,
                    "top_mae": np.nan,
                })
    for col in formation_cols:
        top_col = f"proxy_top_{col}__{fold_col}"
        dist_col = f"proxy_dist_to_{col}__{fold_col}"
        abs_col = f"proxy_abs_dist_to_{col}__{fold_col}"
        data[top_col] = data[top_col].fillna(float(data[f"formation_top_{col}"].median()))
        data[dist_col] = data[top_col] - data["Z"]
        data[abs_col] = np.abs(data[dist_col])
    abs_cols = [f"proxy_abs_dist_to_{col}__{fold_col}" for col in formation_cols]
    data[f"proxy_min_abs_formation_dist__{fold_col}"] = data[abs_cols].min(axis=1)
    data[f"proxy_mean_abs_formation_dist__{fold_col}"] = data[abs_cols].mean(axis=1)

proxy_quality = pd.DataFrame(proxy_quality_rows)
proxy_feature_cols_by_fold = {}
for fold_col in [CONFIG["fold_column"], CONFIG["stress_fold_column"]]:
    proxy_feature_cols_by_fold[fold_col] = [
        *[f"proxy_dist_to_{col}__{fold_col}" for col in formation_cols],
        *[f"proxy_abs_dist_to_{col}__{fold_col}" for col in formation_cols],
        f"proxy_min_abs_formation_dist__{fold_col}",
        f"proxy_mean_abs_formation_dist__{fold_col}",
    ]

proxy_quality_path = TABLE_DIR / "model__xgboost_inference_safe_tuning_proxy_quality__2026-06-28.csv"
proxy_quality.to_csv(proxy_quality_path, index=False)
shutil.copy2(proxy_quality_path, RUN_DIR / proxy_quality_path.name)
proxy_quality.groupby(["fold_column", "formation"]).agg(top_rmse=("top_rmse", "mean"), top_mae=("top_mae", "mean"), n_train_labels=("n_train_labels", "mean")).reset_index()

,fold_column,formation,top_rmse,top_mae,n_train_labels
0,analog_distance_fold,ANCC,40.677052,30.357591,61280.0
1,analog_distance_fold,ASTNL,39.646032,29.447586,61840.0
2,analog_distance_fold,ASTNU,40.868087,30.155400,61840.0
3,analog_distance_fold,BUDA,37.576354,28.123050,61840.0
4,analog_distance_fold,EGFDL,37.249503,28.007169,61760.0
5,analog_distance_fold,EGFDU,37.748049,28.393311,61840.0
6,strict_hybrid_fold,ANCC,38.704697,28.888899,61280.0
7,strict_hybrid_fold,ASTNL,39.593738,29.051101,61840.0
8,strict_hybrid_fold,ASTNU,40.215761,29.691007,61840.0
9,strict_hybrid_fold,BUDA,37.639228,27.734186,61840.0


## Candidate Grid

In [4]:

"Define inference-safe feature sets and tuned XGBoost candidates."

base_params = {
    "tree_method": "hist",
    "random_state": CONFIG["random_state"],
    "n_jobs": 4,
    "verbosity": 0,
    "eval_metric": "rmse",
    "early_stopping_rounds": 50,
}

feature_sets = {
    "safe_base": safe_feature_cols,
    "safe_typewell": [*safe_feature_cols, *typewell_feature_cols],
    "safe_proxy_formations": [*safe_feature_cols, *typewell_feature_cols, *proxy_feature_cols_by_fold[CONFIG["fold_column"]]],
    "oracle_direct_formations": [*safe_feature_cols, *typewell_feature_cols, *direct_formation_feature_cols],
}
feature_sets = {name: list(dict.fromkeys(cols)) for name, cols in feature_sets.items()}

candidates = {
    "pseudo_shallow_public": {
        "objective": "reg:pseudohubererror", "n_estimators": 500, "learning_rate": 0.040, "max_depth": 4,
        "min_child_weight": 120, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.20,
        "reg_lambda": 8.0, "gamma": 0.10,
    },
    "pseudo_shallow_more_trees": {
        "objective": "reg:pseudohubererror", "n_estimators": 700, "learning_rate": 0.030, "max_depth": 4,
        "min_child_weight": 100, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.15,
        "reg_lambda": 7.0, "gamma": 0.05,
    },
    "square_medium_regularized": {
        "objective": "reg:squarederror", "n_estimators": 550, "learning_rate": 0.035, "max_depth": 5,
        "min_child_weight": 90, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.10,
        "reg_lambda": 6.0, "gamma": 0.05,
    },
}

print("feature_sets")
for name, cols in feature_sets.items():
    print(name, len(cols))
print(pd.DataFrame(candidates).T)


feature_sets
safe_base 35
safe_typewell 40
safe_proxy_formations 54
oracle_direct_formations 54
                                      objective n_estimators learning_rate max_depth min_child_weight subsample colsample_bytree reg_alpha reg_lambda gamma
pseudo_shallow_public      reg:pseudohubererror          500          0.04         4              120       0.8              0.8       0.2        8.0   0.1
pseudo_shallow_more_trees  reg:pseudohubererror          700          0.03         4              100      0.85             0.85      0.15        7.0  0.05
square_medium_regularized      reg:squarederror          550         0.035         5               90      0.85             0.85       0.1        6.0  0.05


## Strict-CV Training Loop

In [5]:

"Fit candidate-feature-set combinations on strict_hybrid_fold."


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def xgb_best_iteration(model, fallback):
    value = getattr(model, "best_iteration", None)
    if value is None:
        return int(fallback)
    return int(value) + 1


def xgb_predict(model, X, fallback):
    best_n = xgb_best_iteration(model, fallback)
    return model.predict(X, iteration_range=(0, best_n))


def get_feature_columns(feature_set_name, fold_col):
    cols = feature_sets[feature_set_name]
    if feature_set_name == "safe_proxy_formations" and fold_col != CONFIG["fold_column"]:
        old_cols = proxy_feature_cols_by_fold[CONFIG["fold_column"]]
        new_cols = proxy_feature_cols_by_fold[fold_col]
        mapping = dict(zip(old_cols, new_cols))
        cols = [mapping.get(col, col) for col in cols]
    return [col for col in cols if col in data.columns]

fold_metrics = []
pred_parts = []
importance_parts = []
start_time = time.time()
y_delta = data["target_delta"].astype(float)
y_tvt = data["TVT"].astype(float)
baseline_pred = data["last_known_tvt_feature"].astype(float)

for feature_set_name in ["safe_base", "safe_typewell", "safe_proxy_formations", "oracle_direct_formations"]:
    active_candidates = candidates if feature_set_name != "oracle_direct_formations" else {"pseudo_shallow_public": candidates["pseudo_shallow_public"]}
    feature_cols = get_feature_columns(feature_set_name, CONFIG["fold_column"])
    X_all = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    for candidate, params in active_candidates.items():
        candidate_start = time.time()
        label = f"{candidate}__{feature_set_name}"
        for fold in sorted(data[CONFIG["fold_column"]].unique()):
            valid_mask = data[CONFIG["fold_column"]].eq(fold)
            train_mask = ~valid_mask
            model = XGBRegressor(**base_params, **params)
            model.fit(
                X_all.loc[train_mask], y_delta.loc[train_mask],
                eval_set=[(X_all.loc[valid_mask], y_delta.loc[valid_mask])],
                verbose=False,
            )
            pred_delta = xgb_predict(model, X_all.loc[valid_mask], params["n_estimators"])
            pred_tvt = baseline_pred.loc[valid_mask].to_numpy() + pred_delta
            base = baseline_pred.loc[valid_mask].to_numpy()
            actual = y_tvt.loc[valid_mask].to_numpy()
            fold_metrics.append({
                "candidate": candidate,
                "feature_set": feature_set_name,
                "label": label,
                "fold": int(fold),
                "fold_column": CONFIG["fold_column"],
                "n_train_rows": int(train_mask.sum()),
                "n_valid_rows": int(valid_mask.sum()),
                "n_features": int(len(feature_cols)),
                "best_iteration": xgb_best_iteration(model, params["n_estimators"]),
                "rmse": rmse(actual, pred_tvt),
                "mae": float(mean_absolute_error(actual, pred_tvt)),
                "r2": float(r2_score(actual, pred_tvt)),
                "last_known_hold_rmse": rmse(actual, base),
                "train_time_s": float(time.time() - candidate_start),
            })
            pred_tmp = data.loc[valid_mask, ["wellname", CONFIG["fold_column"], CONFIG["stress_fold_column"], "eval_progress", "selected_analog_distance", "combined_stress_score"]].copy()
            pred_tmp["candidate"] = candidate
            pred_tmp["feature_set"] = feature_set_name
            pred_tmp["label"] = label
            pred_tmp["actual_tvt"] = actual
            pred_tmp["pred_tvt"] = pred_tvt
            pred_tmp["baseline_tvt"] = base
            pred_tmp["residual"] = actual - pred_tvt
            pred_tmp["baseline_residual"] = actual - base
            pred_parts.append(pred_tmp)
            importance_parts.append(pd.DataFrame({"label": label, "candidate": candidate, "feature_set": feature_set_name, "fold": int(fold), "feature": feature_cols, "importance": model.feature_importances_}))

fold_metrics = pd.DataFrame(fold_metrics)
predictions = pd.concat(pred_parts, ignore_index=True)
importance = pd.concat(importance_parts, ignore_index=True)
print(f"elapsed_s={time.time() - start_time:.1f}")
fold_metrics


elapsed_s=399.0


,candidate,feature_set,label,fold,fold_column,n_train_rows,n_valid_rows,n_features,best_iteration,rmse,mae,r2,last_known_hold_rmse,train_time_s
0,pseudo_shallow_public,safe_base,pseudo_shallow_public__safe_base,1,strict_hybrid_fold,61440,31320,35,500,14.518306,9.944525,0.999394,16.418628,5.890634
1,pseudo_shallow_public,safe_base,pseudo_shallow_public__safe_base,2,strict_hybrid_fold,61920,30840,35,500,12.891986,9.267407,0.999588,14.945727,11.872151
2,pseudo_shallow_public,safe_base,pseudo_shallow_public__safe_base,3,strict_hybrid_fold,62160,30600,35,500,13.200005,9.453808,0.999560,15.349476,18.253540
3,pseudo_shallow_more_trees,safe_base,pseudo_shallow_more_trees__safe_base,1,strict_hybrid_fold,61440,31320,35,700,14.491875,9.924477,0.999396,16.418628,8.315058
4,pseudo_shallow_more_trees,safe_base,pseudo_shallow_more_trees__safe_base,2,strict_hybrid_fold,61920,30840,35,698,12.803684,9.189189,0.999594,14.945727,16.479647
5,pseudo_shallow_more_trees,safe_base,pseudo_shallow_more_trees__safe_base,3,strict_hybrid_fold,62160,30600,35,700,13.024285,9.311973,0.999572,15.349476,24.746039
6,square_medium_regularized,safe_base,square_medium_regularized__safe_base,1,strict_hybrid_fold,61440,31320,35,550,13.961552,9.700632,0.999439,16.418628,7.129898
7,square_medium_regularized,safe_base,square_medium_regularized__safe_base,2,strict_hybrid_fold,61920,30840,35,550,13.300430,9.517771,0.999562,14.945727,14.359176
8,square_medium_regularized,safe_base,square_medium_regularized__safe_base,3,strict_hybrid_fold,62160,30600,35,549,13.162182,9.395072,0.999563,15.349476,21.555758
9,pseudo_shallow_public,safe_typewell,pseudo_shallow_public__safe_typewell,1,strict_hybrid_fold,61440,31320,40,500,14.535693,9.950239,0.999392,16.418628,11.555286


## Diagnostics And Exports

In [6]:

"Summarize performance, feature importance, and residual slices."

summary = (
    fold_metrics.groupby(["label", "candidate", "feature_set"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        best_iteration_mean=("best_iteration", "mean"),
        n_features=("n_features", "max"),
        n_valid_rows=("n_valid_rows", "sum"),
    )
    .reset_index()
)
summary["rmse_delta_vs_hold"] = summary["rmse_mean"] - summary["last_known_hold_rmse_mean"]
summary["rmse_delta_vs_public_xgb"] = summary["rmse_mean"] - CONFIG["xgb_public_inference_safe"]
summary["rmse_gap_vs_full_feature_local"] = summary["rmse_mean"] - CONFIG["xgb_full_feature_rmse"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
best_label = str(summary.iloc[0]["label"])
best_candidate = str(summary.iloc[0]["candidate"])
best_feature_set = str(summary.iloc[0]["feature_set"])

predictions["progress_zone"] = pd.cut(predictions["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
predictions["analog_distance_bin"] = pd.qcut(predictions["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

zone_summary = (
    predictions.groupby(["label", "progress_zone"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_residual": float(group["residual"].mean()),
    }), include_groups=False)
    .reset_index()
)
zone_summary["rmse_delta_vs_hold"] = zone_summary["rmse"] - zone_summary["baseline_rmse"]

analog_bin_summary = (
    predictions.groupby(["label", "analog_distance_bin"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "n_wells": group["wellname"].nunique(),
        "mean_selected_analog_distance": float(group["selected_analog_distance"].mean()),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
    }), include_groups=False)
    .reset_index()
)
analog_bin_summary["rmse_delta_vs_hold"] = analog_bin_summary["rmse"] - analog_bin_summary["baseline_rmse"]

well_diag = (
    predictions.loc[predictions["label"].eq(best_label)]
    .groupby("wellname")
    .agg(
        n_rows=("actual_tvt", "size"),
        fold=(CONFIG["fold_column"], "first"),
        stress_fold=(CONFIG["stress_fold_column"], "first"),
        mean_selected_analog_distance=("selected_analog_distance", "mean"),
        mean_combined_stress=("combined_stress_score", "mean"),
        mean_residual=("residual", "mean"),
        mean_abs_residual=("residual", lambda series: float(np.mean(np.abs(series)))),
        rmse=("residual", lambda series: float(np.sqrt(np.mean(np.square(series))))),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

importance_summary = (
    importance.groupby(["label", "candidate", "feature_set", "feature"])["importance"].mean().reset_index()
    .sort_values(["label", "importance"], ascending=[True, False])
)

best_residual_sample = predictions.loc[predictions["label"].eq(best_label)].copy()
if len(best_residual_sample) > 50000:
    best_residual_sample = best_residual_sample.sample(n=50000, random_state=CONFIG["random_state"]).sort_values(["wellname", "eval_progress"])

paths = {
    "summary": TABLE_DIR / "model__xgboost_inference_safe_tuning_summary__2026-06-28.csv",
    "fold_metrics": TABLE_DIR / "model__xgboost_inference_safe_tuning_fold_metrics__2026-06-28.csv",
    "zone_summary": TABLE_DIR / "model__xgboost_inference_safe_tuning_zone_summary__2026-06-28.csv",
    "analog_bin_summary": TABLE_DIR / "model__xgboost_inference_safe_tuning_analog_bin_summary__2026-06-28.csv",
    "well_diagnostics": TABLE_DIR / "model__xgboost_inference_safe_tuning_well_diagnostics__2026-06-28.csv",
    "feature_importance": TABLE_DIR / "model__xgboost_inference_safe_tuning_feature_importance__2026-06-28.csv",
    "residual_sample": TABLE_DIR / "model__xgboost_inference_safe_tuning_residual_sample__2026-06-28.csv",
}
exports = {
    "summary": summary,
    "fold_metrics": fold_metrics,
    "zone_summary": zone_summary,
    "analog_bin_summary": analog_bin_summary,
    "well_diagnostics": well_diag,
    "feature_importance": importance_summary,
    "residual_sample": best_residual_sample,
}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

print(summary.to_string(index=False))
importance_summary.loc[importance_summary["label"].eq(best_label)].head(20)


                                           label                 candidate              feature_set  rmse_mean  rmse_std  mae_mean  r2_mean  last_known_hold_rmse_mean  best_iteration_mean  n_features  n_valid_rows  rmse_delta_vs_hold  rmse_delta_vs_public_xgb  rmse_gap_vs_full_feature_local
 pseudo_shallow_public__oracle_direct_formations     pseudo_shallow_public oracle_direct_formations  12.959650  0.756546  9.273464 0.999555                  15.571277           488.000000          54         92760           -2.611626                 -1.512350                        0.209250
        pseudo_shallow_more_trees__safe_typewell pseudo_shallow_more_trees            safe_typewell  13.417781  0.877855  9.461628 0.999523                  15.571277           699.666667          40         92760           -2.153496                 -1.054219                        0.667381
            pseudo_shallow_more_trees__safe_base pseudo_shallow_more_trees                safe_base  13.439948  0.917649  9.

,label,candidate,feature_set,feature,importance
160,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,horizontal_from_last,0.048392
170,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,md_from_start,0.046527
157,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,extrapolation_stress_score,0.043952
140,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,direct_abs_dist_to_BUDA,0.041432
171,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,row_idx_original,0.039365
146,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,direct_dist_to_BUDA,0.033292
136,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,descriptor_cluster,0.022999
173,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,tvt_linear_proxy,0.022356
135,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,combined_stress_score,0.022229
131,pseudo_shallow_public__oracle_direct_formations,pseudo_shallow_public,oracle_direct_formations,X,0.021664


## Analog-Distance Stress Test

This reruns the best local candidate under the analog-distance split. The purpose is not to pick a new public model directly, but to check whether the chosen recipe survives wells that are deliberately less like the training analogs.

In [7]:
"(run best legal candidate under analog-distance folds)"
legal_summary = summary.loc[~summary["feature_set"].eq("oracle_direct_formations")].sort_values("rmse_mean").reset_index(drop=True)
oracle_summary = summary.loc[summary["feature_set"].eq("oracle_direct_formations")].sort_values("rmse_mean").reset_index(drop=True)
best_row = legal_summary.iloc[0].copy()
best_oracle_row = oracle_summary.iloc[0].copy() if not oracle_summary.empty else None
best_model_key = best_row["label"]
best_feature_set = best_row["feature_set"]
best_candidate_name = best_row["candidate"]
best_candidate = candidates[best_candidate_name]
stress_fold_column = CONFIG["stress_fold_column"]
stress_feature_cols = get_feature_columns(best_feature_set, stress_fold_column)
stress_X = data[stress_feature_cols].replace([np.inf, -np.inf], np.nan)
stress_y_delta = data["target_delta"].astype(float)
stress_y_tvt = data["TVT"].astype(float)
stress_baseline = data["last_known_tvt_feature"].astype(float)

stress_fold_records = []
stress_predictions = pd.Series(index=data.index, dtype=float)

for fold_id in sorted(data[stress_fold_column].dropna().unique()):
    valid_mask = data[stress_fold_column].eq(fold_id)
    train_mask = ~valid_mask
    model = XGBRegressor(**base_params, **best_candidate)
    model.fit(
        stress_X.loc[train_mask],
        stress_y_delta.loc[train_mask],
        eval_set=[(stress_X.loc[valid_mask], stress_y_delta.loc[valid_mask])],
        verbose=False,
    )
    pred_delta = xgb_predict(model, stress_X.loc[valid_mask], best_candidate["n_estimators"])
    fold_pred = stress_baseline.loc[valid_mask].to_numpy() + pred_delta
    stress_predictions.loc[valid_mask] = fold_pred
    stress_fold_records.append({
        "run_id": RUN_ID,
        "model_key": best_model_key,
        "candidate": best_candidate_name,
        "feature_set": best_feature_set,
        "fold_column": stress_fold_column,
        "fold_id": int(fold_id),
        "n_valid": int(valid_mask.sum()),
        "rmse": rmse(stress_y_tvt.loc[valid_mask], fold_pred),
        "mae": float(mean_absolute_error(stress_y_tvt.loc[valid_mask], fold_pred)),
    })

stress_scored = stress_predictions.notna()
stress_fold_df = pd.DataFrame(stress_fold_records)
stress_summary_df = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_key": best_model_key,
    "candidate": best_candidate_name,
    "feature_set": best_feature_set,
    "fold_column": stress_fold_column,
    "n_rows": int(stress_scored.sum()),
    "rmse_mean": rmse(stress_y_tvt.loc[stress_scored], stress_predictions.loc[stress_scored]),
    "mae_mean": float(mean_absolute_error(stress_y_tvt.loc[stress_scored], stress_predictions.loc[stress_scored])),
    "fold_rmse_std": float(stress_fold_df["rmse"].std(ddof=0)),
    "strict_cv_rmse": float(best_row["rmse_mean"]),
    "oracle_strict_cv_rmse": float(best_oracle_row["rmse_mean"]) if best_oracle_row is not None else np.nan,
    "stress_minus_strict_rmse": float(rmse(stress_y_tvt.loc[stress_scored], stress_predictions.loc[stress_scored]) - best_row["rmse_mean"]),
}])

stress_fold_path = TABLE_DIR / "model__xgboost_inference_safe_tuning_analog_distance_fold_metrics__2026-06-28.csv"
stress_summary_path = TABLE_DIR / "model__xgboost_inference_safe_tuning_analog_distance_summary__2026-06-28.csv"
stress_fold_df.to_csv(stress_fold_path, index=False)
stress_summary_df.to_csv(stress_summary_path, index=False)
shutil.copy2(stress_fold_path, RUN_DIR / stress_fold_path.name)
shutil.copy2(stress_summary_path, RUN_DIR / stress_summary_path.name)

stress_summary_df

,run_id,model_key,candidate,feature_set,fold_column,n_rows,rmse_mean,mae_mean,fold_rmse_std,strict_cv_rmse,oracle_strict_cv_rmse,stress_minus_strict_rmse
0,2026-06-28__xgboost_inference_safe_tuning,pseudo_shallow_more_trees__safe_typewell,pseudo_shallow_more_trees,safe_typewell,analog_distance_fold,92760,13.753334,9.800914,0.558506,13.417781,12.95965,0.335553


## Final Report

The report cell writes the experiment conclusion into the artifacts folder and refreshes the model comparison table, so the decision trail stays outside the notebook as well as inside it.

In [8]:
"(write findings and refresh experiment trackers)"
summary_by_feature = summary.groupby("feature_set", as_index=False)["rmse_mean"].min()
safe_base_rmse = float(summary_by_feature.loc[summary_by_feature["feature_set"].eq("safe_base"), "rmse_mean"].min())
best_proxy_rmse = float(summary_by_feature.loc[summary_by_feature["feature_set"].eq("safe_proxy_formations"), "rmse_mean"].min())
best_legal_rmse = float(best_row["rmse_mean"])
best_oracle_rmse = float(best_oracle_row["rmse_mean"]) if best_oracle_row is not None else np.nan
proxy_delta = best_proxy_rmse - safe_base_rmse
legal_delta_vs_base = best_legal_rmse - safe_base_rmse
oracle_gap = best_legal_rmse - best_oracle_rmse if np.isfinite(best_oracle_rmse) else np.nan
stress_rmse = float(stress_summary_df.loc[0, "rmse_mean"])
strict_rmse = best_legal_rmse

if best_feature_set == "safe_proxy_formations" and proxy_delta < -CONFIG["promotion_gate_vs_public"]:
    recommendation = "package_proxy_feature_model"
    recommendation_reason = "Fold-safe proxy formation features improved strict CV beyond the promotion gate."
elif best_feature_set != "safe_base" and legal_delta_vs_base < -CONFIG["promotion_gate_vs_public"]:
    recommendation = "package_best_legal_model"
    recommendation_reason = "A legal non-base feature recipe cleared the local promotion gate."
else:
    recommendation = "do_not_package_yet"
    recommendation_reason = "Best legal tuning improved local strict CV only marginally, while proxy-formation features underperformed safe base."

findings = pd.DataFrame([
    {"finding": "best_legal_model", "value": best_model_key},
    {"finding": "best_legal_strict_rmse", "value": f"{strict_rmse:.4f}"},
    {"finding": "best_legal_stress_rmse", "value": f"{stress_rmse:.4f}"},
    {"finding": "best_legal_feature_set", "value": best_feature_set},
    {"finding": "safe_base_rmse", "value": f"{safe_base_rmse:.4f}"},
    {"finding": "best_proxy_rmse", "value": f"{best_proxy_rmse:.4f}"},
    {"finding": "proxy_minus_base_rmse", "value": f"{proxy_delta:.4f}"},
    {"finding": "oracle_direct_rmse", "value": f"{best_oracle_rmse:.4f}"},
    {"finding": "legal_minus_oracle_rmse", "value": f"{oracle_gap:.4f}"},
    {"finding": "recommendation", "value": recommendation},
    {"finding": "recommendation_reason", "value": recommendation_reason},
])
findings_path = TABLE_DIR / "model__xgboost_inference_safe_tuning_findings__2026-06-28.csv"
findings.to_csv(findings_path, index=False)
shutil.copy2(findings_path, RUN_DIR / findings_path.name)

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison_row = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_family": "xgboost",
    "model_name": "xgboost_inference_safe_tuning",
    "cv_rmse_mean": strict_rmse,
    "cv_rmse_std": float(best_row["rmse_std"]),
    "train_time_s": float(fold_metrics.loc[fold_metrics["label"].eq(best_model_key), "train_time_s"].max()),
    "n_features": int(best_row["n_features"]),
    "notes": recommendation_reason,
    "date": "2026-06-28",
    "validation_design": CONFIG["fold_column"],
    "local_rmse": strict_rmse,
    "stress_rmse": stress_rmse,
    "public_lb_rmse": "",
    "submission_ref": "",
}])
if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    keep_mask = ~(
        comparison_df.get("date", pd.Series(dtype=str)).astype(str).eq("2026-06-28")
        & comparison_df.get("model_name", pd.Series(dtype=str)).astype(str).eq("xgboost_inference_safe_tuning")
    )
    comparison_df = pd.concat([comparison_df.loc[keep_mask], comparison_row], ignore_index=True)
else:
    comparison_df = comparison_row
comparison_df.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / comparison_path.name)

report_lines = [
    "# XGBoost Inference-Safe Tuning Report",
    "",
    f"Run id: `{RUN_ID}`",
    "",
    "## Result",
    "",
    f"- Best legal strict-CV model: `{best_model_key}`",
    f"- Best legal strict-CV RMSE: `{strict_rmse:.4f}`",
    f"- Best legal analog-distance stress RMSE: `{stress_rmse:.4f}`",
    f"- Best legal feature set: `{best_feature_set}`",
    f"- Safe-base RMSE: `{safe_base_rmse:.4f}`",
    f"- Best proxy-formation RMSE: `{best_proxy_rmse:.4f}`",
    f"- Proxy minus base RMSE: `{proxy_delta:.4f}`",
    f"- Oracle direct-formation diagnostic RMSE: `{best_oracle_rmse:.4f}`",
    f"- Legal minus oracle RMSE gap: `{oracle_gap:.4f}`",
    "",
    "## Decision",
    "",
    f"Recommendation: `{recommendation}`.",
    "",
    recommendation_reason,
    "",
    "## Interpretation",
    "",
    "The direct formation feature set remains a diagnostic ceiling only. It confirms formation distance is valuable, but the current fold-safe proxy approach does not recover that signal and should not be packaged as-is.",
    "",
    "## Next Move",
    "",
]
if recommendation == "package_proxy_feature_model":
    report_lines.append("Build a Kaggle package around the proxy-formation feature path and submit it as the next public leaderboard probe.")
elif recommendation == "package_best_legal_model":
    report_lines.append("Package the best legal typewell-enhanced XGBoost recipe as a measured Kaggle probe.")
else:
    report_lines.append("Move to residual stacking or a stronger hidden-test-safe structural proxy instead of submitting this proxy-formation recipe.")

report_path = REPORT_DIR / "model__xgboost_inference_safe_tuning__2026-06-28.md"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

findings

,finding,value
0,best_legal_model,pseudo_shallow_more_trees__safe_typewell
1,best_legal_strict_rmse,13.4178
2,best_legal_stress_rmse,13.7533
3,best_legal_feature_set,safe_typewell
4,safe_base_rmse,13.4399
5,best_proxy_rmse,13.5503
6,proxy_minus_base_rmse,0.1103
7,oracle_direct_rmse,12.9597
8,legal_minus_oracle_rmse,0.4581
9,recommendation,do_not_package_yet
